- 硬件感知
    - (hbm vs. sram) & sm (Streaming Multiprocessor)

### flash attn 与 rope

- 标准流程：显式分离 (Apply RoPE outside Flash Attention)
    - 这是最通用、也是 PyTorch 原生 (torch.nn.functional.scaled_dot_product_attention) 和 Dao-AILab flash-attention 库主要支持的方式。
    - 大致流程
        - 从 HBM 读 $Q, K$ -> SM 计算旋转 -> 写回 HBM 存为 $Q', K'$。
        - 从 HBM 读 $Q', K'$ -> SM 计算 Attention -> 输出。
- 进阶流程：算子融合 (Fused RoPE inside Attention Kernel)
    - 在追求极致推理速度的场景（如 vLLM, TensorRT-LLM, TGI 等推理引擎）中，为了进一步减少显存带宽占用，RoPE 计算会被融合进 Attention Kernel 的加载阶段。
    - 大致流程
        - Kernel 从 HBM 读取原始的 $Q, K$。
        - 在 SRAM (片上缓存) 中，在进行矩阵乘法之前，直接对寄存器中的 $Q, K$ 数据应用旋转。
        - 进行 $Q \cdot K^T$ 计算。

```python
# 1. 生成 Q, K, V
q, k, v = ... 

# 2. 显式应用 RoPE (通常使用 GPU kernel 优化，如 Triton 或 C++)
# q, k 变成了包含了位置信息的 tensor
q_rotated = apply_rope(q, cos, sin)
k_rotated = apply_rope(k, cos, sin)

# 3. 调用 Flash Attention
# flash_attn_func 内部不再处理位置编码，直接做 Q @ K.T
output = flash_attn_func(q_rotated, k_rotated, v)
```